# Data Cleaning - Kairos Project

**Team:** Ilariê

**Objective:** This file details the data cleaning process for Projeto Kairos' datasets. It focus on identify and fix problems like missing values, outliers and inconsistencies, ensuring that the data is in a suitable and reliable format for the next steps of pre-processing and building the predictive model, according to the project constraints.

**Datasets**:
1. SERVICE_ORDER_BASE.xlsx (Service order details)
2. VEHICLES_BASE.xlsx (Vehicle master data)

## 0. Configuration and Data Loading

This initial section establishes the environment, imports the necessary libraries, and loads the datasets that will be used.

### Importing libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Pandas display settings for better visualization
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded successfully!")

### Loading datasets

In [ ]:
try:
    df_service = pd.read_excel('data/SERVICE_ORDER_BASE.xlsx')
    df_vehicles = pd.read_excel('data/VEHICLES_BASE.xlsx')

    datasets = {
        'Service Orders': df_service,
        'Vehicle Master Data': df_vehicles
    }
    print("Datasets loaded successfully!")

except Exception as e:
    print(f"Error on loading datasets: {e}")
    print("Please, ensure the data files are on the correct directory.")

## Initial Data Overview (Optional, can be removed later)

Recommended for a quick check after loading, showing the first few rows and data types.

In [ ]:
for name, df in datasets.items():
    print(f"\n--- Overview: {name} ---")
    display(df.head(3))
    print(df.info())

## 1. Handling Missing Values

**Technical Objective:** Identify and manage null (NaN) values in datasets to ensure data completeness and avoid bias in the analysis and construction of the predictive model.

### 1.1. Detailed Identification of Missing Values

Quantify and calculate the percentage of missing values by column in each DataFrame.

In [ ]:
print("Missing Values by Dataset (Count and Percentage)")
for name, df in datasets.items():
    print(f"\n--- {name} ---")
    missing_data = pd.DataFrame({
        'Null_Count': df.isnull().sum(),
        'Null_Percentage': (df.isnull().sum() / len(df)) * 100
    })
    display(missing_data[missing_data['Null_Count'] > 0].sort_values(by='Null_Percentage', ascending=False))

### 1.2. Value Imputation Strategies

#### For Numeric Columns (Imputation with Median)

The median is preferable to the mean because it is more robust to outliers.

In [ ]:
# Service Orders Dataset
for col in ['GRAND TOTAL', 'PRODUCT QUANTITY', 'UNIT VALUE', 'COUNTER OF SERVICE ORDER']:
    if col in df_service.columns and df_service[col].isnull().any():
        median_value = df_service[col].median()
        df_service[col].fillna(median_value, inplace=True)
        print(f"Service Orders: '{col}' filled with median ({median_value:.2f}).")

# Vehicle Master Data Dataset
if 'MANUFACTURE YEAR' in df_vehicles.columns and df_vehicles['MANUFACTURE YEAR'].isnull().any():
    median_year = df_vehicles['MANUFACTURE YEAR'].median()
    df_vehicles['MANUFACTURE YEAR'].fillna(median_year, inplace=True)
    print(f"Vehicle Master Data: 'MANUFACTURE YEAR' filled with median ({median_year:.0f}).")


#### For Categorical Columns and Identifiers

##### Creating a "Missing" Category

Preserves the information that the value was missing without imputing a value that would have no real meaning.

In [ ]:
# Service Orders Dataset
for col in ['INVOICE', "SUPPLIER'S CODE", "SUPPLIER'S STORE", "NAME OR COMPANY NAME"]:
    if col in df_service.columns and df_service[col].isnull().any():
        df_service[col].fillna('MISSING', inplace=True)
        print(f"Service Orders: '{col}' filled with 'MISSING'.")

##### Filling with Mode

For categorical columns where a new "MISSING" category is not appropriate, filling missing values with the mode (most frequent value) is better.

In [ ]:
# Service Orders Dataset
for col in ["PREVENTIVE_CORRECTIVE MAINTENANCE", "MANUFACTURE YEAR"]:
    if col in df_service.columns and df_service[col].isnull().any():
        mode_value = df_service[col].mode()[0]
        df_service[col].fillna(mode_value, inplace=True)
        print(f"Service Orders: '{col}' filled with mode ({mode_value}).")

# Vehicle Master Dataset
for col in ["MANUFACTURER CODE", "MANUFACTURER NAME"]:
    if col in df_vehicles.columns and df_vehicles[col].isnull().any():
        mode_value = df_vehicles[col].mode()[0]
        df_vehicles[col].fillna(mode_value, inplace=True)
        print(f"Vehicle Master: '{col}' filled with mode ({mode_value}).")

### Final Verification of Missing Values

In [ ]:
print("\nFinal Verification of Missing Values after imputation")
for name, df in datasets.items():
    total_missing = df.isnull().sum().sum()
    print(f"{name}: Total remaining missing values: {total_missing}.")
    if total_missing > 0:
        display(df.isnull().sum()[df.isnull().sum() > 0])

## 2. Handling Outliers (Noisy Values)

**Technical Objective:** Identify and manage extreme data points (outliers) to prevent them from distorting statistical measures and model performance.

### 2.1. Outliers Identification

#### Visualization with Box Plots

Use box plots for quick visual identification of potential outliers.

In [ ]:
def plot_numerical_boxplots(df, title_suffix):
    numerical_cols = df.select_dtypes(include=np.number).columns
    if len(numerical_cols) == 0:
        print(f"No numeric column for {title_suffix}.")
        return

    n_cols = len(numerical_cols)
    n_rows = (n_cols + 2) // 3
    fig, axes = plt.subplots(n_rows, 3, figsize=(18, 5 * n_rows))
    axes = axes.flatten()

    for i, col in enumerate(numerical_cols):
        sns.boxplot(y=df[col], ax=axes[i])
        axes[i].set_title(f'Box Plot of {col} - {title_suffix}')
        axes[i].set_ylabel('')

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

print("Visual Identification of Outliers with Box Plots")
plot_numerical_boxplots(df_service, 'Service Orders')
plot_numerical_boxplots(df_vehicles, 'Vehicle Master Data')

#### IQR (Interquartile Range) Method

For quantitative detection, the IQR method is ideal. Values falling below Q1 - 1.5 * IQR or above Q3 + 1.5 * IQR are considered outliers.

### 2.2. Outlier Treatment Strategies

Blind outlier removal should be avoided. Contextual analysis is crucial. An effective approach is capping, which limits extreme values to the limits calculated by the IQR.

In [ ]:
def apply_iqr_capping(df, col_name, dataset_name):
    if col_name in df.columns and pd.api.types.is_numeric_dtype(df[col_name]):
        Q1 = df[col_name].quantile(0.25)
        Q3 = df[col_name].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers_count = df[(df[col_name] < lower_bound) | (df[col_name] > upper_bound)].shape[0]

        if outliers_count > 0:
            df[col_name] = np.where(df[col_name] < lower_bound, lower_bound, df[col_name])
            df[col_name] = np.where(df[col_name] > upper_bound, upper_bound, df[col_name])
            print(f"{dataset_name}: {outliers_count} outliers in '{col_name}' treated with capping.")
        else:
            print(f"{dataset_name}: No significant outliers found in '{col_name}'.")

print("\nCapping Application (Outliers Treatment)")

# Service Orders Dataset
apply_iqr_capping(df_service, 'GRAND TOTAL', 'Service Orders')
apply_iqr_capping(df_service, 'PRODUCT QUANTITY', 'Service Orders')
apply_iqr_capping(df_service, 'UNIT VALUE', 'Service Orders')
apply_iqr_capping(df_service, 'COUNTER OF SERVICE ORDER', 'Service Orders')

# Vehicle Master Data Dataset
apply_iqr_capping(df_vehicles, 'MANUFACTURE YEAR', 'Vehicle Master Data')